## **Project 1 Phase 1**
Rylee Sandifer and Cheney Floyd

**Business Problem and What we hope to achieve:**

The business problem is that many borrowers may not repay their loans and may choose to default, causing the bank to lose money. We hope to achieve a successful model that uses variables including characteristics and statistics of potential borrowers to determine what matters: whether they will default on their loan. The target column tells us whether or not the client paid off the loan or not. A 1 means that the client had payment difficulties, and a 0 means the client did not. We hope to reduce false negative predictions so that we reduce the amount of defaulters approved for loans.

In [1]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve

from sklearn.preprocessing import StandardScaler


In [4]:
df = pd.read_csv("data/application_train.csv")

In the cell below we decided to first drop all columns that are missing more than 50% of the data. We did this because they are missing too much data/information to provide any useful analysis or predictions. They are also missing too much data to impute or fill in the data, as that would cause bias and disrupt the original distribution.

In [5]:
missing = df.isnull().sum()/len(df)
overfiftymissing = missing[missing > 0.5].index.tolist()

In [6]:
threshold = int(len(df) * 0.5) + 1
df_cleaned = df.dropna(axis=1, thresh=threshold)

We then decided to drop all columns that contained "FLAG_DOCUMENT". These columns only told us if a client included a certain document, and we concluded they were not significant. Each column was also 99% 0s, so they didn't give us much information for analysis.

In [7]:
cols = df_cleaned.columns.tolist()
cols
for col in cols:
    if "FLAG_DOCUMENT" in col:
        df_cleaned = df_cleaned.drop(col, axis=1)

We then used binary encoding to encode the gender, contract type, car owner, and house owner columns in order to use them in our logistic regression model. We also dropped the 4 rows that contained 'XNA' in the gender column as they were proportionately insignifant and were prohibiting us from using binary encoding.

In [8]:
df_cleaned = df_cleaned[df_cleaned['CODE_GENDER']!='XNA']
df_cleaned['CODE_GENDER'] = df_cleaned['CODE_GENDER'].map({'M': 1, 'F': 0})
df_cleaned['NAME_CONTRACT_TYPE'] = df_cleaned['NAME_CONTRACT_TYPE'].map({'Cash loans': 1, 'Revolving loans': 0})
df_cleaned['FLAG_OWN_CAR'] = df_cleaned['FLAG_OWN_CAR'].map({'N': 1, 'Y': 0})
df_cleaned['FLAG_OWN_REALTY'] = df_cleaned['FLAG_OWN_REALTY'].map({'N': 1, 'Y': 0})

From initial analysis by looking at our data, and collaborating with AI, we have decided that our missing data was MCAR, and used median imputation to fill in the missing data. We are not 100% finished developing this part, so it will be continued in phase 2 as we create a logistic regression model to decide if the data is MCAR, MAR, or MNAR.

We are planning to attempt to combine the "EXT_SOURCE_2" and "EXT_SOURCE_3" columns into summary statistics such as mean, median, minimum, and maximum in further data processing in Phase 2 of our project.

The following cells is where we do the median imputation for each variable.

In [9]:
for col in ["EXT_SOURCE_2", "EXT_SOURCE_3"]:
    df_cleaned[f"{col}_MISSING"] = df_cleaned[col].isna().astype(int)
    df_cleaned[col] = df_cleaned[col].fillna(df_cleaned[col].median())

In [10]:
df_cleaned["DAYS_LAST_PHONE_CHANGE_MISSING"] = (
    df_cleaned["DAYS_LAST_PHONE_CHANGE"].isna().astype(int)
)

# Median imputation
df_cleaned["DAYS_LAST_PHONE_CHANGE"] = (
    df_cleaned["DAYS_LAST_PHONE_CHANGE"]
    .fillna(df_cleaned["DAYS_LAST_PHONE_CHANGE"].median())
)

In [11]:
df_cleaned["AMT_GOODS_PRICE"] = (
    df_cleaned["AMT_GOODS_PRICE"]
    .fillna(df_cleaned["AMT_GOODS_PRICE"].median())
)

In [12]:
df_cleaned["FLOORSMAX_AVG_MISSING"] = (
    df_cleaned["FLOORSMAX_AVG"].isna().astype(int)
)

# Median imputation
df_cleaned["FLOORSMAX_AVG"] = (
    df_cleaned["FLOORSMAX_AVG"]
    .fillna(df_cleaned["FLOORSMAX_AVG"].median())
)

In [13]:
df_cleaned["FLOORSMAX_MEDI_MISSING"] = (
    df_cleaned["FLOORSMAX_MEDI"].isna().astype(int)
)

# Median imputation
df_cleaned["FLOORSMAX_MEDI"] = (
    df_cleaned["FLOORSMAX_MEDI"]
    .fillna(df_cleaned["FLOORSMAX_MEDI"].median())
)

In [14]:
df_cleaned["FLOORSMAX_MODE_MISSING"] = (
    df_cleaned["FLOORSMAX_MODE"].isna().astype(int)
)

# Median imputation
df_cleaned["FLOORSMAX_MODE"] = (
    df_cleaned["FLOORSMAX_MODE"]
    .fillna(df_cleaned["FLOORSMAX_MODE"].median())
)

Through collaboration with AI, we concluded the following columns to be the most important and significant columns in the data when predicting default. We kept the following columns for our logistic regression model.

In [15]:
full_keep=["TARGET",
      "EXT_SOURCE_3",
      "EXT_SOURCE_3_MISSING",
      "EXT_SOURCE_2",
      "EXT_SOURCE_2_MISSING", 
      "DAYS_BIRTH",
      "DAYS_ID_PUBLISH",
      "DAYS_REGISTRATION",
      "DAYS_EMPLOYED",
      "DAYS_LAST_PHONE_CHANGE",
      "DAYS_LAST_PHONE_CHANGE_MISSING",
      "FLAG_EMP_PHONE",
      "REGION_RATING_CLIENT",
      "REGION_RATING_CLIENT_W_CITY",
      "REGION_POPULATION_RELATIVE",
      "REG_CITY_NOT_WORK_CITY",
      "REG_CITY_NOT_LIVE_CITY",
      "AMT_GOODS_PRICE",
      "FLOORSMAX_AVG",
      "FLOORSMAX_AVG_MISSING",
      "FLOORSMAX_MEDI",
      "FLOORSMAX_MEDI_MISSING",
      "FLOORSMAX_MODE",
      "FLOORSMAX_MODE_MISSING"]
df_cleaned=df_cleaned[full_keep]

We then built our logistic regression model. We first created our X and y sets by dropping the target column from our cleaned data frame. The target column tells us whether or not the client paid off the loan or not. A 1 means that the client had payment difficulties, and a 0 means the client did not.

We performed a 70/30 train test split with our data and stratified on y to keep the datasets proportionate. We then scaled the data with StandardScaler after the split. For our model we used class_weight="balanced" to deal with the class imbalance that we see in our data.

In [16]:
X = df_cleaned.drop('TARGET', axis=1)
y = df_cleaned['TARGET']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=1, stratify=y)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression(solver='lbfgs', max_iter=1000, class_weight="balanced", C=3) 
model.fit(X_train_scaled, y_train)

predictions = model.predict(X_test_scaled)
probabilities = model.predict_proba(X_test_scaled)[:,1]

print(confusion_matrix(y_test, predictions))
print(classification_report(y_test, predictions))

print(roc_auc_score(y_test, probabilities))

[[56840 27965]
 [ 2533  4915]]
              precision    recall  f1-score   support

           0       0.96      0.67      0.79     84805
           1       0.15      0.66      0.24      7448

    accuracy                           0.67     92253
   macro avg       0.55      0.67      0.52     92253
weighted avg       0.89      0.67      0.74     92253

0.7240018565368672


**Reflection on our Results**
- Class 1 precision = 0.15. This shows us that when we predict that a customer is going to have payment problems, we are wrong 85% of the time. 
- Class 0 precision = 0.96. This is extrememly good and shows us that when we predict that a customer is going to be a good one, we are correct 96% of the time.
- We see that we have a lot of class imbalance. This is something we need to ensure we take care of in our model
- Class 1 recall = 0.66. This means that we catch 66% of the positives. We prevent 2/3 of the bad customers from getting a loan.
- Class 0 recall = 0.67. This means that we catch 67% of the negatives. This is not very good considering they are the much larger class.
- Accuracy here does not mean too much to us. This is because a bad model that predicted 0 100% of the time would still get 92% accuracy.

The main takeaway from our results is that our model likes to predict positive a lot. This is somewhat helpful because in our case, missing a positive is much more costly than missing a negative. So while we may be rejecting a lot of good customers, we are accepting less bad customers.


**Plan for Phase 2:**

In the next two weeks, we will focus on interpreting whether missing data is MAR, MNAR, or MCAR and we will look to use another logistic regression model to predict MAR. We will focus on joining the main table with supplementary tables to add variables to improve the results of our model so that we are able to accurately predict whether someone will default more than 2/3 of the time (which is what our current model does). We are also planning to attempt to combine the "EXT_SOURCE_2" and "EXT_SOURCE_3" columns into summary statistics such as mean, median, minimum, and maximum in further data processing in Phase 2 of our project.

**AI Reflection:**

We used ChatGPT to help us with code and ideas for our current model. It was helpful in the logistic regression code, but not as helpful with the imputation methods because it didn't consider MAR, MNAR, and MCAR, so we are going to be revising our current imputation methods to try to improve our missing data and improve our model. Overall, AI was helpful in getting started with our model and very helpful in determining which columns were likely to be more significant than others.